# Data Cleaning Notebook

This notebook documents the cleaning workflow for the scraped used-cars dataset. It starts with loading and inspection, then fixes missing values, converts numeric fields, extracts location features, and finishes with validation checks.


## 1. Import Required Libraries

Load the libraries used for dataframe handling, numerical operations, plotting, and regular expressions.


In [78]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import re

## 2. Optional Combined Dataset Loading

This cell keeps an earlier option for loading both new and used cars together. The code is currently commented out, so it does not affect the notebook output.


In [79]:

#new_cars = pd.read_csv(r"data\raw\hatla2ee_new_cars.csv")
#used_cars = pd.read_csv(r"data\raw\hatla2ee_used_cars.csv")

#df = pd.concat([new_cars, used_cars], ignore_index=True)



## 3. Load Used Cars Dataset

Read the raw used-cars CSV file and store it in the dataframe `df`.


In [80]:
df = pd.read_csv(r"data\raw\hatla2ee_used_cars.csv")


## 4. Preview Dataset Size And First Rows

Print the dataset shape and display the first rows to understand the raw structure.


In [81]:
print(df.shape)
df.head()

(10173, 14)


,page,car_type,car_name,year,mileage,transmission,fuel,price,location,brand,model,car_url,image_url,scraped_at
0,1,used,Mercedes C 180 2025,2025,"5,000 KM",Automatic,Gas,"3,300,000 EGP","Tagamo3 - New Cairo, Cairo",Mercedes,C 180,https://eg.hatla2ee.com/en/car/mercedes/c-180/...,https://legion-images.hatla2ee.com/listing_ima...,2026-04-30 14:20:32
1,1,used,BMW X5 M 2015,2015,"180,000 KM",Automatic,Gas,"1,750,000 EGP","Sheikh Zayed City, Giza",BMW,X5 M,https://eg.hatla2ee.com/en/car/bmw/X5-M/7210739,https://legion-images.hatla2ee.com/listing_ima...,2026-04-30 14:20:34
2,1,used,Ford EcoSport 2019,2019,"124,999 KM",Automatic,Gas,"850,000 EGP","Heliopolis, Cairo",Ford,EcoSport,https://eg.hatla2ee.com/en/car/ford/ecosport/7...,https://legion-images.hatla2ee.com/listing_ima...,2026-04-30 14:20:36
3,1,used,BMW Z4 2022,2022,"20,000 KM",Automatic,Gas,"3,700,000 EGP","Sheikh Zayed City, Giza",BMW,Z4,https://eg.hatla2ee.com/en/car/bmw/z4/7210388,https://legion-images.hatla2ee.com/listing_ima...,2026-04-30 14:20:36
4,1,used,Mercedes GLA 200 2020,2020,"19,000 KM",Automatic,Gas,"2,000,000 EGP","Sheikh Zayed City, Giza",Mercedes,GLA 200,https://eg.hatla2ee.com/en/car/mercedes/GLA-20...,https://legion-images.hatla2ee.com/listing_ima...,2026-04-30 14:20:37


## 5. Inspect Columns And Data Types

Use `df.info()` to review column names, non-null counts, data types, and memory usage.


In [82]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10173 entries, 0 to 10172
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   page          10173 non-null  int64 
 1   car_type      10173 non-null  object
 2   car_name      10173 non-null  object
 3   year          10173 non-null  int64 
 4   mileage       10173 non-null  object
 5   transmission  10173 non-null  object
 6   fuel          10173 non-null  object
 7   price         10173 non-null  object
 8   location      10173 non-null  object
 9   brand         10173 non-null  object
 10  model         10171 non-null  object
 11  car_url       10173 non-null  object
 12  image_url     10173 non-null  object
 13  scraped_at    10173 non-null  object
dtypes: int64(2), object(12)
memory usage: 1.1+ MB


## 6. Check Missing Values

Count missing values in every column to identify fields that need cleaning.


In [83]:
df.isnull().sum()

page            0
car_type        0
car_name        0
year            0
mileage         0
transmission    0
fuel            0
price           0
location        0
brand           0
model           2
car_url         0
image_url       0
scraped_at      0
dtype: int64

## 7. Check Full Duplicate Rows

Check whether any rows are exact duplicates across all columns.


In [84]:
df.duplicated().sum()

0

## 8. Inspect Missing Model Records

Display the records where `model` is missing before applying any manual fix.


In [85]:
df[df['model'].isna()]

,page,car_type,car_name,year,mileage,transmission,fuel,price,location,brand,model,car_url,image_url,scraped_at
4577,235,used,Nissan Sunny 2015,2015,"185,000 KM",Automatic,Gas,"490,000 EGP",Nissan,Sunny,NaN,https://eg.hatla2ee.com/en/car/nissan/sunny/71...,https://legion-images.hatla2ee.com/listing_ima...,2026-04-30 15:36:38
5886,322,used,BMW X6 2019,2019,"90,000 KM",Automatic,Gas,"2,800,000 EGP",BMW,X6,NaN,https://eg.hatla2ee.com/en/car/bmw/x6/7197351,https://legion-images.hatla2ee.com/listing_ima...,2026-04-30 16:02:21


## 9. Fix Missing Model Records

Fill the two missing model values and correct their related brand and location values.


In [86]:
# fixing the nulls in model column
df.loc[4577, ['model', 'brand', 'location']] = ['Sunny', 'Nissan', '']
df.loc[5886, ['model', 'brand', 'location']] = ['X6', 'BMW', '']


## 10. Confirm Missing Model Fix

Run `df.info()` again to confirm that the model column no longer has missing values.


In [87]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10173 entries, 0 to 10172
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   page          10173 non-null  int64 
 1   car_type      10173 non-null  object
 2   car_name      10173 non-null  object
 3   year          10173 non-null  int64 
 4   mileage       10173 non-null  object
 5   transmission  10173 non-null  object
 6   fuel          10173 non-null  object
 7   price         10173 non-null  object
 8   location      10173 non-null  object
 9   brand         10173 non-null  object
 10  model         10173 non-null  object
 11  car_url       10173 non-null  object
 12  image_url     10173 non-null  object
 13  scraped_at    10173 non-null  object
dtypes: int64(2), object(12)
memory usage: 1.1+ MB


## 11. Preview Price Values

Look at sample `price` values before converting the column from text to numbers.


In [88]:
df["price"].head()

0    3,300,000 EGP
1    1,750,000 EGP
2      850,000 EGP
3    3,700,000 EGP
4    2,000,000 EGP
Name: price, dtype: object

## 12. Preview Mileage Values

Look at sample `mileage` values before converting the column from text to numbers.


In [89]:
df["mileage"].head()

0      5,000 KM
1    180,000 KM
2    124,999 KM
3     20,000 KM
4     19,000 KM
Name: mileage, dtype: object

## 13. Convert Price And Mileage

Remove non-numeric characters from `price` and `mileage`, then convert both columns to numeric values.


In [90]:
# transform the columns "price" & "mileage"
df["price"] = pd.to_numeric(df["price"].str.replace(r"\D", "", regex=True))
df["mileage"] = pd.to_numeric(df["mileage"].str.replace(r"\D", "", regex=True))

## 14. Drop Scraping Page Column

Remove the `page` column because it is only related to scraping pagination, not the car itself.


In [91]:
df.drop(columns=["page"], inplace=True)

## 15. Check Data After Numeric Cleaning

Inspect the dataframe again after converting numeric columns and dropping `page`.


In [92]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10173 entries, 0 to 10172
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   car_type      10173 non-null  object
 1   car_name      10173 non-null  object
 2   year          10173 non-null  int64 
 3   mileage       10173 non-null  int64 
 4   transmission  10173 non-null  object
 5   fuel          10173 non-null  object
 6   price         10173 non-null  int64 
 7   location      10173 non-null  object
 8   brand         10173 non-null  object
 9   model         10173 non-null  object
 10  car_url       10173 non-null  object
 11  image_url     10173 non-null  object
 12  scraped_at    10173 non-null  object
dtypes: int64(3), object(10)
memory usage: 1.0+ MB


## 16. Preview Cleaned Rows

Display the first rows after the initial cleaning steps.


In [93]:
df.head()

,car_type,car_name,year,mileage,transmission,fuel,price,location,brand,model,car_url,image_url,scraped_at
0,used,Mercedes C 180 2025,2025,5000,Automatic,Gas,3300000,"Tagamo3 - New Cairo, Cairo",Mercedes,C 180,https://eg.hatla2ee.com/en/car/mercedes/c-180/...,https://legion-images.hatla2ee.com/listing_ima...,2026-04-30 14:20:32
1,used,BMW X5 M 2015,2015,180000,Automatic,Gas,1750000,"Sheikh Zayed City, Giza",BMW,X5 M,https://eg.hatla2ee.com/en/car/bmw/X5-M/7210739,https://legion-images.hatla2ee.com/listing_ima...,2026-04-30 14:20:34
2,used,Ford EcoSport 2019,2019,124999,Automatic,Gas,850000,"Heliopolis, Cairo",Ford,EcoSport,https://eg.hatla2ee.com/en/car/ford/ecosport/7...,https://legion-images.hatla2ee.com/listing_ima...,2026-04-30 14:20:36
3,used,BMW Z4 2022,2022,20000,Automatic,Gas,3700000,"Sheikh Zayed City, Giza",BMW,Z4,https://eg.hatla2ee.com/en/car/bmw/z4/7210388,https://legion-images.hatla2ee.com/listing_ima...,2026-04-30 14:20:36
4,used,Mercedes GLA 200 2020,2020,19000,Automatic,Gas,2000000,"Sheikh Zayed City, Giza",Mercedes,GLA 200,https://eg.hatla2ee.com/en/car/mercedes/GLA-20...,https://legion-images.hatla2ee.com/listing_ima...,2026-04-30 14:20:37


## 17. Prepare Location Matching Helpers

Define a simple governorate keyword dictionary and helper functions to clean location text and identify governorates.


In [94]:
# Simple keyword map: if any keyword appears in location, use its governorate.
governorate_keywords = {
    "cairo": ["cairo", "new cairo", "tagamo3", "nasr city", "heliopolis", "maadi", "mokattam", "shorouk", "madinaty", "rehab"],
    "giza": ["giza", "6th of october", "october", "sheikh zayed", "zayed", "haram", "faisal", "dokki", "mohandessin"],
    "alexandria": ["alex", "alexandria"],
    "dakahlia": ["dakahlia", "mansoura"],
    "red sea": ["red sea", "hurghada", "gouna"],
    "beheira": ["beheira", "damanhur"],
    "fayoum": ["fayoum", "faiyum"],
    "gharbia": ["gharbia", "tanta", "mahalla"],
    "ismailia": ["ismailia"],
    "monufia": ["monufia", "menofia", "shebin", "shibin", "ashmun", "ashmoun", "bagour", "tala", "quwaysna"],
    "minya": ["minya"],
    "qalyubia": ["qalyubia", "qalubia", "benha", "shubra el kheima"],
    "new valley": ["new valley", "wadi gedid", "kharga", "farafra"],
    "suez": ["suez"],
    "aswan": ["aswan", "kom ombo"],
    "assiut": ["assiut", "assyut", "asyut"],
    "beni suef": ["beni suef", "ben suef"],
    "port said": ["port said", "portsaid"],
    "damietta": ["damietta", "domyat"],
    "sharkia": ["sharkia", "sharqia", "zagazig", "faqous", "bilbeis", "abu kabir", "dyarb negm"],
    "south sinai": ["south sinai", "sharm", "dahab"],
    "north sinai": ["north sinai", "arish"],
    "kafr el sheikh": ["kafr el sheikh"],
    "matrouh": ["matrouh", "marsa matrouh"],
    "luxor": ["luxor"],
    "qena": ["qena"],
    "sohag": ["sohag"],
}

def clean_location(location):
    if pd.isna(location):
        return pd.NA
    location = str(location).lower().strip()
    location = re.sub(r"[^a-z0-9,\s]", " ", location)
    location = re.sub(r"\s+", " ", location)
    return location

def get_governorate(location):
    if pd.isna(location):
        return pd.NA
    for governorate, keywords in governorate_keywords.items():
        if any(keyword in location for keyword in keywords):
            return governorate
    return pd.NA

## 18. Extract City And Governorate

Create `location_clean`, extract `city`, and assign `governorate` from the location text.


In [95]:
df["location_clean"] = df["location"].apply(clean_location)

# Most locations look like: "city, governorate".
df["city"] = df["location_clean"].str.split(",").str[0].str.strip()
df["governorate"] = df["location_clean"].apply(get_governorate)

df["city"] = df["city"].replace("", pd.NA)

df[["location", "governorate", "city"]].head(10)

,location,governorate,city
0,"Tagamo3 - New Cairo, Cairo",cairo,tagamo3 new cairo
1,"Sheikh Zayed City, Giza",giza,sheikh zayed city
2,"Heliopolis, Cairo",cairo,heliopolis
3,"Sheikh Zayed City, Giza",giza,sheikh zayed city
4,"Sheikh Zayed City, Giza",giza,sheikh zayed city
5,"Minya Al Qamh, Sharqia",minya,minya al qamh
6,Alexandria,alexandria,alexandria
7,"Tagamo3 - New Cairo, Cairo",cairo,tagamo3 new cairo
8,"Tagamo3 - New Cairo, Cairo",cairo,tagamo3 new cairo
9,"Talkha, Dakahlia",dakahlia,talkha


## 19. Check Suspicious Numeric Values

Count records with invalid or suspicious `price`, `mileage`, or `year` values.


In [96]:
current_year = pd.Timestamp.today().year

suspicious_prices = df[df["price"].isna() | (df["price"] <= 0)]
suspicious_mileage = df[df["mileage"].isna() | (df["mileage"] < 0)]
suspicious_years = df[df["year"].isna() | (df["year"] < 0) | (df["year"] > current_year + 1)]

len(suspicious_prices), len(suspicious_mileage), len(suspicious_years)

(0, 0, 0)

## 20. Final Validation Checks

Collect key quality checks into one dictionary, including row count, duplicates, missing values, and unmatched locations.


In [97]:
validation_checks = {
    "rows": len(df),
    "duplicate_rows": int(df.duplicated().sum()),
    "missing_brand": int(df["brand"].isna().sum()),
    "missing_model": int(df["model"].isna().sum()),
    "missing_price": int(df["price"].isna().sum()),
    "missing_mileage": int(df["mileage"].isna().sum()),
    "missing_year": int(df["year"].isna().sum()),
    "unmatched_locations": int(df["location"].isna().sum()),
}

validation_checks

{'rows': 10173,
 'duplicate_rows': 0,
 'missing_brand': 0,
 'missing_model': 0,
 'missing_price': 0,
 'missing_mileage': 0,
 'missing_year': 0,
 'unmatched_locations': 0}

## 21. Empty Follow-Up Cell

This empty code cell can be used later for exporting the cleaned dataset or adding extra checks.


## Summary

What was done:

- Loaded the raw used-cars dataset.
- Reviewed shape, sample rows, data types, and missing values.
- Checked for fully duplicate rows.
- Found two rows with missing `model` values.
- Fixed the missing `model`, `brand`, and `location` values.
- Converted `price` and `mileage` from text into numeric columns.
- Dropped the scraping-only `page` column.
- Created location helper functions.
- Extracted `city` and `governorate` from `location`.
- Checked suspicious numeric values.
- Built a final validation dictionary.

Main result: the dataset is cleaner and more ready for analysis, with fixed model values, numeric price and mileage, and added location features.
